In [ ]:
pip install sentence-transformers rouge-score nltk scikit-learn --quiet


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 59.0 MB/s eta 0:00:00


In [ ]:
# Re-download NLTK punkt tokenizer data (force re-install)
import nltk
nltk.download('punkt', force=True)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
# === Install Required Libraries ===
!pip install -q spacy sentence-transformers rouge-score nltk
!python -m spacy download en_core_web_sm

# === Import Modules ===
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
from tqdm import tqdm
import nltk
nltk.download('punkt')

# === Load Uploaded File ===
df = pd.read_csv("merged_cleaned_text_PaddleOCR2.csv")
texts = df['text'].dropna().tolist()

# === SpaCy Sentence Chunking ===
nlp = spacy.load("en_core_web_sm")

def sentence_chunk(text, max_words=120, overlap=20):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]
    chunks, chunk, length = [], [], 0
    for sent in sentences:
        words = sent.split()
        if length + len(words) > max_words:
            chunks.append(" ".join(chunk))
            chunk = chunk[-overlap:] if overlap > 0 else []
            length = sum(len(s.split()) for s in chunk)
        chunk.append(sent)
        length += len(words)
    if chunk:
        chunks.append(" ".join(chunk))
    return chunks

# === Create Chunks ===
all_chunks = []
for doc_id, text in enumerate(tqdm(texts)):
    for i, chunk in enumerate(sentence_chunk(text)):
        all_chunks.append({
            "doc_id": doc_id,
            "chunk_id": i,
            "chunk_text": chunk
        })

print(f"✅ Total chunks: {len(all_chunks)}")

# === Embedding Chunks ===
model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [c["chunk_text"] for c in all_chunks]
embeddings = model.encode(chunk_texts, show_progress_bar=True, device='cuda' if model.device.type == 'cuda' else 'cpu')

# === Calculate Metrics Between Overlapping Chunks ===
rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
bleu_scores, rouge1_scores, rougeL_scores, cosine_scores = [], [], [], []

for i in tqdm(range(len(all_chunks) - 1)):
    if all_chunks[i]['doc_id'] == all_chunks[i + 1]['doc_id']:
        text1 = all_chunks[i]["chunk_text"]
        text2 = all_chunks[i + 1]["chunk_text"]

        # Cosine Similarity
        cos_sim = cosine_similarity(
            embeddings[i].reshape(1, -1),
            embeddings[i + 1].reshape(1, -1)
        )[0][0]
        cosine_scores.append(cos_sim)

        # BLEU Score
        bleu = sentence_bleu(
            [text1.split()],
            text2.split(),
            smoothing_function=SmoothingFunction().method1
        )
        bleu_scores.append(bleu)

        # ROUGE Scores
        scores = rouge.score(text1, text2)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

# === Print Results ===
print("\n🔍 Average Chunk Overlap Metrics:")
print(f"✅ Average Cosine Similarity: {np.mean(cosine_scores):.4f}")
print(f"✅ Average BLEU Score:         {np.mean(bleu_scores):.4f}")
print(f"✅ Average ROUGE-1 F1:         {np.mean(rouge1_scores):.4f}")
print(f"✅ Average ROUGE-L F1:         {np.mean(rougeL_scores):.4f}")


  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
100%|██████████| 9100/9100 [13:23<00:00, 11.33it/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Total chunks: 163109


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5098 [00:00<?, ?it/s]

100%|██████████| 163108/163108 [1:08:40<00:00, 39.58it/s]


🔍 Average Chunk Overlap Metrics:
✅ Average Cosine Similarity: 0.9737
✅ Average BLEU Score:         0.9101
✅ Average ROUGE-1 F1:         0.9439
✅ Average ROUGE-L F1:         0.9406
